# Reto Final · Análisis exploratorio de capturas de *Anopheles*

**Curso:** Aprendizaje Estadístico 2026-2  
**Estudiantes:** Jeronimo Lopez Gomez – Jose Daniel Godoy

**Pregunta:** ¿Cuántos individuos de *Anopheles* se podrían capturar en una campaña de cuatro noches en Cheté y con qué incertidumbre?

**Referencia inicial (Fermi):** 346 *Anopheles*; escenarios hipotéticos de 122 a 605. Estos valores no son intervalos estadísticos.

Este notebook **organiza el EDA existente** siguiendo `plan_analisis.md`. No se afirma haber entrenado ni evaluado un modelo predictivo. Para reproducirlo, deben estar disponibles `occurrence.txt`, `dhime_mensual.csv` y `dhime_diario.csv` en el directorio de trabajo.

## 1. Preparación

Importamos librerías y configuramos las gráficas. **Pregunta:** ¿qué herramientas necesitamos para reproducir el análisis?

In [ ]:
import numpy as np
import pandas as pd
import os
print('Directorio de trabajo:', os.getcwd())

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import contextily as cx #Real Maps Background
import geopandas as gpd
import seaborn as sns
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D
import matplotlib as mpl
from matplotlib.patches import Patch


# ------------------------------------------------------------
# PALETA DE COLORES
# ------------------------------------------------------------

COLORES = {
    'principal': '#4472A8',       # Azul
    'secundario': '#C07836',      # Naranja
    'terciario': '#6B8E83',       # Verde grisáceo
    'acento': '#8064A2',          # Violeta
    'negativo': '#B65C5C',        # Rojo suave
    'gris': '#7F8790',
    'gris_claro': '#D9DEE5',
    'fondo': '#FFFFFF',
    'texto': '#303841'
}

# ------------------------------------------------------------
# CONFIGURACIÓN GLOBAL
# ------------------------------------------------------------

mpl.rcParams.update({

    # Figura
    'figure.figsize': (12, 6),
    'figure.dpi': 110,
    'savefig.dpi': 300,
    'figure.facecolor': COLORES['fondo'],
    'axes.facecolor': COLORES['fondo'],

    # Tipografía
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'DejaVu Sans'],
    'font.size': 10,
    'text.color': COLORES['texto'],

    # Títulos
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.titlecolor': COLORES['texto'],
    'axes.titlepad': 16,

    # Etiquetas
    'axes.labelsize': 11,
    'axes.labelcolor': COLORES['texto'],

    # Ejes
    'axes.edgecolor': '#BFC5CC',
    'axes.linewidth': 0.8,
    'axes.spines.top': False,
    'axes.spines.right': False,

    # Cuadrícula
    'axes.grid': True,
    'grid.color': '#D9DEE5',
    'grid.linestyle': '--',
    'grid.linewidth': 0.7,
    'grid.alpha': 0.65,

    # Marcas de los ejes
    'xtick.color': COLORES['texto'],
    'ytick.color': COLORES['texto'],
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,

    # Leyendas
    'legend.fontsize': 9,
    'legend.frameon': True,
    'legend.facecolor': 'white',
    'legend.edgecolor': '#D9DEE5',
    'legend.framealpha': 0.95,

    # Líneas
    'lines.linewidth': 2,
    'lines.markersize': 5
})


# ============================================================
# FUNCIÓN PARA FORMATEAR TODAS LAS GRÁFICAS
# ============================================================

def formato_grafica(
    ax,
    titulo=None,
    xlabel=None,
    ylabel=None,
    grid_axis='y'
):

    if titulo:
        ax.set_title(titulo, pad=18, fontweight='bold')

    if xlabel:
        ax.set_xlabel(xlabel, labelpad=9)

    if ylabel:
        ax.set_ylabel(ylabel, labelpad=9)

    ax.grid(
        axis=grid_axis,
        linestyle='--',
        linewidth=0.7,
        alpha=0.55,
        color=COLORES['gris_claro']
    )

    ax.set_axisbelow(True)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    ax.spines['left'].set_color('#BFC5CC')
    ax.spines['bottom'].set_color('#BFC5CC')

    ax.tick_params(
        direction='out',
        length=4,
        width=0.8
    )

    return ax


**Interpretación / resultados a completar después de ejecutar**  


## 2. Carga y descripción del conjunto de datos

**Pregunta:** ¿cuántos registros y variables contiene Darwin Core? Revisamos sus primeras filas y la información faltante.

In [ ]:
# ============================================================
# 1. CARGA DEL DATASET
# ============================================================

df = pd.read_csv(
    './occurrence.txt',
    sep='\t',
    encoding='utf-8',
    low_memory=False
)


# ============================================================
# 2. INSPECCIÓN GENERAL DEL DATASET
# ============================================================

print("DATASET DE ESPECIES DEL GÉNERO ANOPHELES")
print(f"Dimensiones del dataset: {df.shape[0]} registros × {df.shape[1]} columnas")

display(df.head(5))


# ============================================================
# 3. ANÁLISIS DE ESTRUCTURA Y CALIDAD DE LOS DATOS
# ============================================================

description = pd.DataFrame({
    'Tipo de dato': df.dtypes,
    'Valores no nulos': df.notna().sum(),
    'Valores nulos': df.isna().sum(),
    '% Nulos': (df.isna().mean() * 100).round(2),
    'Valores únicos': df.nunique()
})

print("INFORMACIÓN GENERAL POR COLUMNAS")

display(description.sort_values('% Nulos', ascending=False))

**Interpretación / resultados a completar después de ejecutar**  
**Cómo reportarlo:** anotar número de filas/columnas y porcentaje de valores faltantes de variables relevantes.

## 3. Calidad y consistencia

**Pregunta:** ¿hay identificadores repetidos, coordenadas inválidas o fechas incompletas? Una ocurrencia no implica necesariamente un individuo.

In [ ]:
# ============================================================
# 4. VALIDACIONES DE INTEGRIDAD DEL DATASET
# ============================================================

print("VALIDACIONES DE INTEGRIDAD")

# Registros completamente duplicados
print("Registros duplicados:", df.duplicated().sum())

# Identificadores de ocurrencia
if 'occurrenceID' in df.columns:
    print("occurrenceID nulos:", df['occurrenceID'].isna().sum())
    print("occurrenceID duplicados:", df['occurrenceID'].duplicated().sum())

# Columnas completamente vacías
empty_columns = df.columns[df.isna().all()].tolist()
print("Columnas completamente vacías:", empty_columns)

# Columnas con un único valor
constant_columns = df.columns[df.nunique() <= 1].tolist()
print("Columnas constantes:", constant_columns)


# ============================================================
# 5. VALIDACIÓN DE COORDENADAS GEOGRÁFICAS
# ============================================================

if {'decimalLatitude', 'decimalLongitude'}.issubset(df.columns):

    latitude = pd.to_numeric(df['decimalLatitude'], errors='coerce')
    longitude = pd.to_numeric(df['decimalLongitude'], errors='coerce')

    valid_coordinates = (
        latitude.between(-90, 90) &
        longitude.between(-180, 180)
    )

    print("VALIDACIÓN GEOGRÁFICA")
    print("Coordenadas faltantes:", (latitude.isna() | longitude.isna()).sum())
    print("Coordenadas fuera de rango:", (~valid_coordinates & latitude.notna() & longitude.notna()).sum())


# ============================================================
# 6. VALIDACIÓN Y PROCESAMIENTO DE FECHAS
# ============================================================

if 'eventDate' in df.columns:

    # Separar fechas iniciales y finales
    date_parts = df['eventDate'].str.split('/', n=1, expand=True)

    dates_start = pd.to_datetime(date_parts[0], errors='coerce')
    dates_end = pd.to_datetime(date_parts[1], errors='coerce')

    # Identificar registros con fecha individual o intervalo
    is_interval = date_parts[1].notna()

    print("VALIDACIÓN TEMPORAL")
    print("Registros con fecha individual:", (~is_interval & dates_start.notna()).sum())
    print("Registros con intervalo de fechas:", (is_interval & dates_start.notna() & dates_end.notna()).sum())
    print("Fechas faltantes o inválidas:", dates_start.isna().sum())

    print("Fecha inicial mínima:", dates_start.min())
    print("Fecha final máxima:", dates_end.fillna(dates_start).max())

**Interpretación / resultados a completar después de ejecutar**  
**Cómo reportarlo:** señalar duplicados, fechas de intervalo y coordenadas faltantes. No interpretar la falta de un registro como captura cero.

## 4. Cobertura temporal, localidades, esfuerzo y especies

**Preguntas:** ¿cuándo y dónde se hicieron capturas?, ¿qué especies aparecen?, ¿cómo fue el esfuerzo de muestreo? Los valores de `samplingEffort` describen un protocolo y no prueban horas efectivas por cada mosquito.

In [ ]:
# ============================================================
# 7. EXPLORACIÓN TEMPORAL DEL DATASET
# ============================================================

# No sobrescribir eventDate: puede contener intervalos inicio/fin.
fecha_evento_inicio = df['eventDate'].astype('string').str.split('/', n=1).str[0]
df['fecha_inicio_registro'] = pd.to_datetime(
    fecha_evento_inicio, errors='coerce', utc=True
)

fecha_inicio = df['fecha_inicio_registro'].min()
fecha_fin = df['fecha_inicio_registro'].max()

print("PERIODO TEMPORAL DE LOS REGISTROS")
print("-" * 50)
print(f"Fecha inicial: {fecha_inicio.strftime('%Y-%m-%d')}")
print(f"Fecha final:   {fecha_fin.strftime('%Y-%m-%d')}")
print(f"Duración:      {(fecha_fin - fecha_inicio).days} días")
print(f"Años cubiertos: {fecha_inicio.year} - {fecha_fin.year}")
print(f"Total de registros: {len(df)}")
print("-" * 50)


# ============================================================
# 8. EXPLORACIÓN DE LOS LUGARES Y HORARIOS DE CAPTURA
# ============================================================

print("Lugares de captura según el hábitat:")
print(df['habitat'].value_counts(dropna=False))
print("--" * 50)

print("\nHorarios o periodos de captura:")
print(df['eventTime'].value_counts(dropna=False))
print("--" * 50)


# ============================================================
# 9. DISTRIBUCIÓN GEOGRÁFICA DE LOS REGISTROS
# ============================================================

print("Clasificación por municipio y localidad:")

locations = (
    df.groupby(['municipality', 'locality'], dropna=False)
      .size()
      .reset_index(name='registros')
      .sort_values('registros', ascending=False)
)

display(locations)


# ============================================================
# 10. EXPLORACIÓN DEL ESFUERZO DE MUESTREO
# ============================================================

print("Descripción del esfuerzo de muestreo:")
print(df['samplingEffort'].value_counts(dropna=False).head(20))
print("--" * 50)

# Extracción de horas y noches
df['horas_noche'] = pd.to_numeric(
    df['samplingEffort'].str.extract(
        r'(?i)(\d+(?:[.,]\d+)?)\s*h(?:oras?)?'
    )[0].str.replace(',', '.', regex=False),
    errors='coerce'
)

df['num_noches'] = pd.to_numeric(
    df['samplingEffort'].str.extract(
        r'(?i)(\d+(?:[.,]\d+)?)\s*noches?'
    )[0].str.replace(',', '.', regex=False),
    errors='coerce'
)

print("\nNúmero de noches por registro:")
print(df['num_noches'].value_counts(dropna=False).sort_index())

print("\nHoras por noche:")
print(df['horas_noche'].value_counts(dropna=False).sort_index())


# ============================================================
# 11. COMPOSICIÓN TAXONÓMICA
# ============================================================

print("Especies de Anopheles registradas:")

species = (
    df['scientificName']
      .value_counts(dropna=False)
      .rename_axis('especie')
      .reset_index(name='registros')
)

species['porcentaje'] = (
    species['registros'] / len(df) * 100
).round(2)

display(species)

**Interpretación / resultados a completar después de ejecutar**  
**Cómo reportarlo:** describir fechas, localidades, especies y esfuerzo. Verificar manualmente la extracción de horas y noches.

## 5. Distribución geográfica

**Pregunta:** ¿dónde están los registros georreferenciados y cuáles pertenecen a Cheté? Los puntos del mapa representan registros o cantidades según la gráfica, no necesariamente viviendas únicas.

In [ ]:
# ============================================================
# 11. DISTRIBUCIÓN GEOGRÁFICA DE ANOPHELES EN TIMBIQUÍ
# ============================================================

# ============================================================
# 11.1. PREPARACIÓN DE LOS DATOS
# ============================================================

# Filtrar registros con coordenadas válidas
df_mapa = df.dropna(
    subset=['decimalLatitude', 'decimalLongitude']
).copy()

df_mapa['decimalLatitude'] = pd.to_numeric(
    df_mapa['decimalLatitude'], errors='coerce'
)

df_mapa['decimalLongitude'] = pd.to_numeric(
    df_mapa['decimalLongitude'], errors='coerce'
)

df_mapa['individualCount'] = pd.to_numeric(
    df_mapa['individualCount'], errors='coerce'
)

df_mapa = df_mapa.dropna(
    subset=['decimalLatitude', 'decimalLongitude']
)

# Filtrar municipio
#df_mapa = df_mapa[
#    df_mapa['municipality'].str.strip().str.casefold() == 'timbiquí'
#].copy()

# Revisar disponibilidad de cantidades
print("Registros con cantidad de individuos:")
print("Informados:", df_mapa['individualCount'].notna().sum())
print("Sin información:", df_mapa['individualCount'].isna().sum())

# Convertir a GeoDataFrame
gdf_mapa = gpd.GeoDataFrame(
    df_mapa,
    geometry=gpd.points_from_xy(
        df_mapa['decimalLongitude'],
        df_mapa['decimalLatitude']
    ),
    crs='EPSG:4326'
).to_crs(epsg=3857)

# Separar registros con y sin cantidad de individuos
gdf_cantidad = gdf_mapa[
    gdf_mapa['individualCount'].notna()
].copy()

gdf_sin_cantidad = gdf_mapa[
    gdf_mapa['individualCount'].isna()
].copy()

# Identificar registros de Cheté
gdf_chete = gdf_mapa[
    gdf_mapa['locality'].str.strip().str.casefold() == 'cheté'
].copy()

gdf_chete_cantidad = gdf_chete[
    gdf_chete['individualCount'].notna()
].copy()

gdf_chete_sin_cantidad = gdf_chete[
    gdf_chete['individualCount'].isna()
].copy()


# ============================================================
# 11.2. ESCALA DE COLOR Y TAMAÑO
# ============================================================

cmap = plt.get_cmap('YlOrRd')

if not gdf_cantidad.empty:

    vmin = gdf_cantidad['individualCount'].min()
    vmax = gdf_cantidad['individualCount'].max()

    norm = Normalize(vmin=vmin, vmax=vmax)

    max_log = np.log1p(
        gdf_cantidad['individualCount'].clip(lower=0)
    ).max()

    def calcular_tamanos(datos):

        valores = np.log1p(
            datos['individualCount'].clip(lower=0)
        )

        if max_log == 0:
            return np.full(len(datos), 100)

        return 80 + 220 * valores / max_log


# ============================================================
# 11.3. MAPA MUNICIPAL DE TIMBIQUÍ
# ============================================================

fig1, ax1 = plt.subplots(
    figsize=(6, 10)
)

# Extensión geográfica del municipio
xmin, ymin, xmax, ymax = gdf_mapa.total_bounds

margin_x = (xmax - xmin) * 0.08
margin_y = (ymax - ymin) * 0.06

ax1.set_xlim(xmin - margin_x, xmax + margin_x)
ax1.set_ylim(ymin - margin_y, ymax + margin_y)

# Registros sin cantidad informada
if not gdf_sin_cantidad.empty:

    gdf_sin_cantidad.plot(
        ax=ax1,
        color='#00BCEB',
        markersize=65,
        alpha=0.9,
        edgecolor='black',
        linewidth=0.6,
        label='Cantidad no informada',
        zorder=3
    )

# Registros con cantidad informada
if not gdf_cantidad.empty:

    gdf_cantidad.plot(
        ax=ax1,
        color=gdf_cantidad['individualCount'].map(
            lambda x: cmap(norm(x))
        ),
        markersize=calcular_tamanos(gdf_cantidad),
        alpha=0.9,
        edgecolor='black',
        linewidth=0.7,
        zorder=4
    )

# Destacar ubicación de Cheté
if not gdf_chete.empty:

    punto_chete = gdf_chete.geometry.unary_union.centroid

    ax1.scatter(
        punto_chete.x,
        punto_chete.y,
        color='cyan',
        marker='X',
        s=150,
        edgecolor='black',
        linewidth=1,
        label='Localidad de Cheté',
        zorder=5
    )

# Mapa base
cx.add_basemap(
    ax1,
    source=cx.providers.Esri.WorldImagery,
    zoom=11
)

ax1.set_title(
    'Distribución espacial de registros de Anopheles\n'
    'Municipios de Timbiquí y Guapi, Cauca',
    fontsize=15,
    fontweight='bold',
    pad=15
)

ax1.legend(
    loc='lower left',
    fontsize=10,
    framealpha=0.95
)

ax1.set_axis_off()

# Barra de color
if not gdf_cantidad.empty:

    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])

    cbar1 = fig1.colorbar(
        sm,
        ax=ax1,
        orientation='horizontal',
        fraction=0.035,
        pad=0.035,
        shrink=0.65,
        aspect=40
    )

    cbar1.set_label(
        'Individuos registrados por ocurrencia',
        fontsize=10
    )

    cbar1.ax.tick_params(labelsize=9)

fig1.subplots_adjust(
    left=0.04,
    right=0.96,
    top=0.94,
    bottom=0.08
)

plt.show()


# ============================================================
# 11.4. DETALLE DE LA LOCALIDAD DE CHETÉ
# ============================================================

if not gdf_chete.empty:

    fig2, ax2 = plt.subplots(
        figsize=(9, 4)
    )

    # Extensión geográfica de Cheté
    xmin, ymin, xmax, ymax = gdf_chete.total_bounds

    margin_x = max((xmax - xmin) * 0.65, 700)
    margin_y = max((ymax - ymin) * 0.65, 700)

    ax2.set_xlim(
        xmin - margin_x,
        xmax + margin_x
    )

    ax2.set_ylim(
        ymin - margin_y,
        ymax + margin_y
    )

    # Registros sin cantidad informada
    if not gdf_chete_sin_cantidad.empty:

        gdf_chete_sin_cantidad.plot(
            ax=ax2,
            color='#00BCEB',
            markersize=65,
            alpha=0.9,
            edgecolor='black',
            linewidth=0.6,
            label='Cantidad no informada',
            zorder=3
        )

    # Registros con cantidad informada
    if not gdf_chete_cantidad.empty:

        gdf_chete_cantidad.plot(
            ax=ax2,
            color=gdf_chete_cantidad['individualCount'].map(
                lambda x: cmap(norm(x))
            ),
            markersize=calcular_tamanos(gdf_chete_cantidad),
            alpha=0.9,
            edgecolor='black',
            linewidth=0.7,
            zorder=4
        )

    # Mapa base
    cx.add_basemap(
        ax2,
        source=cx.providers.Esri.WorldImagery,
        zoom=16
    )

    ax2.set_title(
        'Distribución espacial de registros de Anopheles\n'
        'Localidad de Cheté, Timbiquí',
        fontsize=15,
        fontweight='bold',
        pad=15
    )

    if not gdf_chete_sin_cantidad.empty:

        ax2.legend(
            loc='lower left',
            fontsize=10,
            framealpha=0.95
        )

    ax2.set_axis_off()

    # Barra de color
    if not gdf_cantidad.empty:

        sm = ScalarMappable(norm=norm, cmap=cmap)
        sm.set_array([])

        cbar2 = fig2.colorbar(
            sm,
            ax=ax2,
            orientation='horizontal',
            fraction=0.04,
            pad=0.045,
            shrink=0.55,
            aspect=40
        )

        cbar2.set_label(
            'Individuos registrados por ocurrencia',
            fontsize=10
        )

        cbar2.ax.tick_params(labelsize=9)

    fig2.subplots_adjust(
        left=0.04,
        right=0.96,
        top=0.92,
        bottom=0.10
    )

    plt.show()

else:

    print("No hay registros georreferenciados asociados a Cheté.")


# ============================================================
# 11.5. RESUMEN
# ============================================================

print("\nRESUMEN DE DISTRIBUCIÓN GEOGRÁFICA")
print("-" * 50)

print("Registros cartografiados:", len(gdf_mapa))
print("Registros asociados a Cheté:", len(gdf_chete))
print("Registros con cantidad informada:", len(gdf_cantidad))
print("Registros sin cantidad informada:", len(gdf_sin_cantidad))

**Interpretación / resultados a completar después de ejecutar**  
**Cómo reportarlo:** distinguir número de registros, número de individuos y ubicaciones únicas.

## 6. Datos pluviométricos IDEAM

**Pregunta:** ¿qué estaciones, periodos y unidades tenemos disponibles? Revisamos cobertura y registros repetidos.

In [ ]:
# ============================================================
# 12.  DATOS PLUVIOMÉTRICOS DEL IDEAM
# ============================================================

lluvia = pd.read_csv(
    './dhime_mensual.csv',
    sep=',',
    encoding='utf-8',
    low_memory=False,
    parse_dates=['Fecha']
)
# ============================================================
# 12.1. IDENTIFICACIÓN DE ESTACIONES Y VARIABLES
# ============================================================

print("ESTACIONES METEOROLÓGICAS")
print("-" * 60)

estaciones = (
    lluvia.groupby(
        ['CodigoEstacion', 'NombreEstacion'],
        dropna=False
    )
    .agg(
        registros=('Valor', 'count'),
        fecha_inicial=('Fecha', 'min'),
        fecha_final=('Fecha', 'max'),
        precipitacion_maxima=('Valor', 'max')
    )
    .reset_index()
    .sort_values('registros', ascending=False)
)

display(estaciones)

print("\nVARIABLES REGISTRADAS")
print(lluvia['Variable'].value_counts(dropna=False))

print("\nPARÁMETROS REGISTRADOS")
print(lluvia['Parametro'].value_counts(dropna=False))

print("\nUNIDADES DE MEDIDA")
print(lluvia['Unidad'].value_counts(dropna=False))

print("\nNIVELES DE APROBACIÓN")
print(lluvia['NivelAprobacion'].value_counts(dropna=False))


# ============================================================
# 12.2. DISTRIBUCIÓN TEMPORAL DE LOS REGISTROS
# ============================================================

lluvia['Año'] = lluvia['Fecha'].dt.year
lluvia['Mes'] = lluvia['Fecha'].dt.month
lluvia['NombreMes'] = lluvia['Fecha'].dt.month_name()

print("\nREGISTROS POR AÑO")
display(
    lluvia['Año'].value_counts().sort_index()
)

print("\nREGISTROS POR MES")
display(
    lluvia['Mes'].value_counts().sort_index()
)


# ============================================================
# 12.3. REVISIÓN DE REGISTROS REPETIDOS
# ============================================================

duplicados = lluvia.duplicated(
    subset=['CodigoEstacion', 'Fecha', 'Variable', 'Parametro'],
    keep=False
)

print("\nREGISTROS REPETIDOS POR ESTACIÓN Y FECHA")
print("Total:", duplicados.sum())

if duplicados.any():
    display(
        lluvia.loc[duplicados].sort_values(
            ['CodigoEstacion', 'Fecha']
        ).head(20)
    )


**Interpretación / resultados a completar después de ejecutar**  
**Cómo reportarlo:** indicar qué estaciones y años cubren los registros; documentar que no son mediciones dentro de las viviendas.

## 7. Precipitación mensual

**Pregunta:** ¿cómo cambia la lluvia entre meses y estaciones? Esta comparación es descriptiva, sin atribuir la lluvia de una estación a cada vivienda.

In [ ]:
# ============================================================
# 13. PRECIPITACIÓN MENSUAL PROMEDIO POR ESTACIÓN (2021-2023)
# ============================================================

lluvia = lluvia.copy()  # Continuar con el archivo mensual cargado antes

lluvia['Fecha'] = pd.to_datetime(
    lluvia['Fecha'], errors='coerce'
)

lluvia['Valor'] = pd.to_numeric(
    lluvia['Valor'], errors='coerce'
)

lluvia = lluvia.dropna(
    subset=['Fecha', 'Valor', 'CodigoEstacion']
).copy()

lluvia['Año'] = lluvia['Fecha'].dt.year
lluvia['Mes'] = lluvia['Fecha'].dt.month

# Filtrar registros de 2021, 2022 y 2023
# y excluir la estación Gorgona
lluvia = lluvia[
    (lluvia['Año'].isin([2021, 2022, 2023])) &
    (~lluvia['NombreEstacion'].str.contains(
        'Gorgona', case=False, na=False
    ))
].copy()

# Nombres de los meses
meses = [
    'Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun',
    'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic'
]

# ============================================================
# ESTADÍSTICAS MENSUALES POR ESTACIÓN
# ============================================================

stats = (
    lluvia.groupby(
        ['CodigoEstacion', 'NombreEstacion', 'Mes']
    )['Valor']
    .mean()
    .reset_index(name='mean')
)

# ============================================================
# PROMEDIO GENERAL DE TODAS LAS ESTACIONES
# ============================================================

stats_general = (
    lluvia.groupby('Mes')['Valor']
    .agg(['mean', 'std', 'count'])
    .reindex(range(1, 13))
)

# ============================================================
# GRÁFICA
# ============================================================

fig, ax = plt.subplots(figsize=(14, 6))

# Paleta de colores para las estaciones
estaciones = stats[
    ['CodigoEstacion', 'NombreEstacion']
].drop_duplicates()

colores = plt.cm.tab20(
    np.linspace(0, 1, len(estaciones))
)

# Curvas individuales por estación
for (_, estacion), color in zip(
    estaciones.iterrows(), colores
):

    datos = stats[
        stats['CodigoEstacion'] == estacion['CodigoEstacion']
    ].sort_values('Mes')

    ax.plot(
        datos['Mes'],
        datos['mean'],
        marker='o',
        markersize=3,
        linewidth=1.5,
        color=color,
        alpha=0.55
    )

# Promedio general
ax.plot(
    stats_general.index,
    stats_general['mean'],
    marker='o',
    markersize=7,
    linewidth=3,
    color=COLORES['principal'],
    zorder=10,
    label='Promedio general'
)

# Desviación estándar del promedio general
ax.fill_between(
    stats_general.index,
    stats_general['mean'] - stats_general['std'],
    stats_general['mean'] + stats_general['std'],
    color=COLORES['principal'],
    alpha=0.12,
    zorder=5
)

# ============================================================
# FORMATO
# ============================================================

ax.set_xticks(range(1, 13))
ax.set_xticklabels(meses)

formato_grafica(
    ax,
    titulo='Precipitación mensual promedio\n'
           'Timbiquí y Guapi, Cauca | DHIME-IDEAM (2021-2023)',
    xlabel='Mes',
    ylabel='Precipitación promedio (mm)'
)

# ============================================================
# LEYENDA GENERAL
# ============================================================

handles_general = [
    Line2D(
        [0], [0],
        color=COLORES['principal'],
        linewidth=3,
        marker='o',
        markersize=6,
        label='Promedio general'
    ),
    Patch(
        facecolor=COLORES['principal'],
        alpha=0.12,
        label='± Desviación estándar'
    )
]

legend_general = ax.legend(
    handles=handles_general,
    title='Estadísticas generales',
    loc='upper left'
)

ax.add_artist(legend_general)

# ============================================================
# LEYENDA DE ESTACIONES
# ============================================================

handles_estaciones = [
    Line2D(
        [0], [0],
        color=color,
        linewidth=2,
        alpha=0.75,
        label=estacion['NombreEstacion']
    )
    for (_, estacion), color in zip(
        estaciones.iterrows(), colores
    )
]

ax.legend(
    handles=handles_estaciones,
    title='Estaciones meteorológicas',
    bbox_to_anchor=(1.02, 1),
    loc='upper left'
)

# ============================================================
# AJUSTE FINAL
# ============================================================

plt.tight_layout()
plt.show()

**Interpretación / resultados a completar después de ejecutar**  
**Cómo reportarlo:** describir estacionalidad sin afirmar que la lluvia causa variaciones de abundancia.

## 8. Capturas registradas por mes

**Pregunta:** ¿cómo varía la cantidad registrada por mes y cuántas fechas de muestreo existen? Las fechas distintas no sustituyen el esfuerzo en horas-colector.

In [ ]:
# ============================================================
# 14. DISTRIBUCIÓN MENSUAL DE CAPTURAS Y ESFUERZO DE MUESTREO
# ============================================================

# 1. PREPARAR DATOS
# ============================================================

df_mosquitos = df.copy()

col_conteo = (
    'individualCount'
    if 'individualCount' in df_mosquitos.columns
    else 'occurrenceID'
)

if col_conteo == 'individualCount':
    df_mosquitos[col_conteo] = pd.to_numeric(
        df_mosquitos[col_conteo],
        errors='coerce'
    )  # Mantener como NaN los conteos faltantes; no asumir 1
else:
    df_mosquitos[col_conteo] = 1

# Procesar fechas
df_mosquitos['eventDate'] = pd.to_datetime(
    df_mosquitos['eventDate'].astype('string').str.split('/', n=1).str[0],
    errors='coerce',
    utc=True
)

df_mosquitos = df_mosquitos.dropna(
    subset=['eventDate']
).copy()

df_mosquitos['Mes_Num'] = df_mosquitos['eventDate'].dt.month

# ============================================================
# 2. AGRUPAR CAPTURAS Y ESFUERZO DE MUESTREO
# ============================================================

datos_mensuales = (
    df_mosquitos.groupby('Mes_Num')
    .agg(
        Total_Mosquitos=(col_conteo, 'sum'),
        Dias_Muestreados=('eventDate', 'nunique')
    )
    .reindex(range(1, 13), fill_value=0)
    .reset_index()
)

meses = [
    'Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun',
    'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic'
]

# ============================================================
# 3. CONFIGURAR GRÁFICA: BARRAS AGRUPADAS
# ============================================================

fig, ax1 = plt.subplots(figsize=(14, 6))

ancho = 0.35
x = datos_mensuales['Mes_Num']

# Eje 1: Individuos registrados
ax1.bar(
    x - ancho / 2,
    datos_mensuales['Total_Mosquitos'],
    width=ancho,
    color=COLORES['principal'],
    edgecolor='white',
    linewidth=0.8,
    alpha=0.85,
    label='Individuos registrados',
    zorder=3
)

ax1.set_xlabel('Mes')
ax1.set_ylabel('Número de individuos registrados')

ax1.set_xticks(range(1, 13))
ax1.set_xticklabels(meses)

# Eje 2: Esfuerzo de muestreo
ax2 = ax1.twinx()

ax2.bar(
    x + ancho / 2,
    datos_mensuales['Dias_Muestreados'],
    width=ancho,
    color=COLORES['secundario'],
    edgecolor='white',
    linewidth=0.8,
    alpha=0.85,
    label='Días de muestreo',
    zorder=3
)

ax2.set_ylabel('Días de muestreo')

ax2.set_ylim(
    0,
    max(1, datos_mensuales['Dias_Muestreados'].max() * 1.2)
)

# ============================================================
# 4. FORMATO Y TÍTULO
# ============================================================

formato_grafica(
    ax1,
    titulo=(
        'Distribución mensual de capturas y esfuerzo de muestreo\n'
        'Anopheles | 2021-2023'
    ),
    xlabel='Mes',
    ylabel='Número de individuos registrados'
)

ax1.tick_params(
    axis='y',
    labelcolor=COLORES['principal']
)

ax2.tick_params(
    axis='y',
    labelcolor=COLORES['secundario']
)

ax2.yaxis.label.set_color(COLORES['secundario'])

ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(True)
ax2.spines['right'].set_color('#BFC5CC')

# ============================================================
# 5. LEYENDA
# ============================================================

handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    handles1 + handles2,
    labels1 + labels2,
    loc='upper left'
)

plt.tight_layout()
plt.show()

# ============================================================
# 6. TABLA DE RESULTADOS
# ============================================================

print('DISTRIBUCIÓN MENSUAL: CAPTURAS Y ESFUERZO DE MUESTREO')
print('-' * 60)

tabla_mensual = datos_mensuales[
    ['Mes_Num', 'Total_Mosquitos', 'Dias_Muestreados']
].copy()

tabla_mensual['Mes'] = tabla_mensual['Mes_Num'].map(
    dict(enumerate(meses, start=1))
)

display(
    tabla_mensual[
        ['Mes', 'Total_Mosquitos', 'Dias_Muestreados']
    ]
)

print('-' * 60)

**Interpretación / resultados a completar después de ejecutar**  
**Cómo reportarlo:** comparar capturas y número de fechas por mes. Un mes con más registros puede haber tenido más esfuerzo.

## 9. Capturas por fecha y precipitación mensual

**Pregunta:** ¿existe un patrón descriptivo entre capturas y precipitación? El cociente «mosquitos/fecha» no equivale a mosquitos/hora-colector.

In [ ]:
# ============================================================
# 15. TASA DE CAPTURA NORMALIZADA VS PRECIPITACIÓN
# ============================================================

# ============================================================
# 1. PROCESAMIENTO Y NORMALIZACIÓN DE MOSQUITOS
# ============================================================

df_mosquitos = df.copy()

# Asegurar columna de conteo
col_conteo = (
    'individualCount'
    if 'individualCount' in df_mosquitos.columns
    else 'occurrenceID'
)

if col_conteo == 'individualCount':

    df_mosquitos[col_conteo] = pd.to_numeric(
        df_mosquitos[col_conteo],
        errors='coerce'
    )  # No imputar conteos desconocidos como 1

else:
    df_mosquitos[col_conteo] = 1

# Procesar fechas Darwin Core
# Se utiliza la fecha inicial cuando eventDate contiene intervalos.

partes_fecha = df_mosquitos['eventDate'].astype(
    'string'
).str.split('/', n=1, expand=True)

df_mosquitos['Fecha_Inicio'] = pd.to_datetime(
    partes_fecha[0],
    errors='coerce',
    utc=True
)

df_mosquitos = df_mosquitos.dropna(
    subset=['Fecha_Inicio']
).copy()

# Filtrar registros correspondientes a 2021, 2022 y 2023
df_mosquitos = df_mosquitos[
    df_mosquitos['Fecha_Inicio'].dt.year.isin(
        [2021, 2022, 2023]
    )
].copy()

df_mosquitos['Mes_Num'] = (
    df_mosquitos['Fecha_Inicio'].dt.month.astype(int)
)

# ============================================================
# 2. AGRUPACIÓN MENSUAL DE CAPTURAS Y ESFUERZO
# ============================================================

datos_mensuales = (
    df_mosquitos.groupby('Mes_Num')
    .agg(
        Total_Mosquitos=(col_conteo, 'sum'),
        Dias_Muestreados=('Fecha_Inicio', 'nunique')
    )
    .reindex(range(1, 13), fill_value=0)
    .reset_index()
)

# ============================================================
# 3. TASA DE CAPTURA NORMALIZADA
# ============================================================

datos_mensuales['Mosquitos_Por_Fecha'] = np.divide(
    datos_mensuales['Total_Mosquitos'],
    datos_mensuales['Dias_Muestreados'],
    out=np.full(12, np.nan),
    where=datos_mensuales['Dias_Muestreados'].values > 0
)

# ============================================================
# 4. INTEGRACIÓN CON PRECIPITACIÓN (IDEAM)
# ============================================================

# La variable lluvia ya está filtrada para 2021-2023
# y excluye la estación Gorgona desde la sección 13.

precipitacion = (
    lluvia.groupby('Mes')['Valor']
    .mean()
    .reindex(range(1, 13))
)

datos_mensuales['Precipitacion_Promedio_mm'] = (
    precipitacion.values
)

# Nombres de los meses
meses = [
    'Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun',
    'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic'
]

datos_mensuales['Mes_Nombre'] = meses

# ============================================================
# 5. GRÁFICA COMBINADA: TASA DE CAPTURA VS PRECIPITACIÓN
# ============================================================

fig, ax1 = plt.subplots(figsize=(14, 6))

# Eje principal: tasa de captura
ax1.bar(
    datos_mensuales['Mes_Num'],
    datos_mensuales['Mosquitos_Por_Fecha'],
    width=0.60,
    color=COLORES['principal'],
    label='Tasa de captura',
    zorder=3
)

ax1.set_xticks(range(1, 13))
ax1.set_xticklabels(meses)

formato_grafica(
    ax1,
    titulo=(
        'Tasa mensual de captura de Anopheles y precipitación promedio\n'
        'Timbiquí y Guapi, Cauca | 2021-2023'
    ),
    xlabel='Mes',
    ylabel='Tasa de captura promedio mensual'
)

ax1.tick_params(
    axis='y',
    labelcolor=COLORES['principal']
)

ax1.yaxis.label.set_color(COLORES['principal'])

# Eje secundario: precipitación
ax2 = ax1.twinx()

ax2.plot(
    datos_mensuales['Mes_Num'],
    datos_mensuales['Precipitacion_Promedio_mm'],
    color=COLORES['secundario'],
    marker='o',
    markersize=7,
    linewidth=2.8,
    label='Precipitación promedio',
    zorder=5
)

ax2.set_ylabel(
    'Precipitación promedio mensual (mm)',
    color=COLORES['secundario'],
    labelpad=9
)

ax2.tick_params(
    axis='y',
    labelcolor=COLORES['secundario']
)

ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(True)
ax2.spines['right'].set_color('#BFC5CC')

# Leyenda conjunta
handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    handles1 + handles2,
    labels1 + labels2,
    loc='upper left'
)

plt.tight_layout()
plt.show()

# ============================================================
# 6. TABLA DE RESULTADOS
# ============================================================

print("\n" + "=" * 85)
print("TASA DE CAPTURA NORMALIZADA VS PRECIPITACIÓN")
print("=" * 85)

tabla_final = datos_mensuales[
    [
        'Mes_Nombre',
        'Dias_Muestreados',
        'Total_Mosquitos',
        'Mosquitos_Por_Fecha',
        'Precipitacion_Promedio_mm'
    ]
].copy()

tabla_final.columns = [
    'Mes',
    'Días muestreados',
    'Total mosquitos',
    'Mosquitos / fecha',
    'Precipitación (mm)'
]

display(tabla_final.round(2))

**Interpretación / resultados a completar después de ejecutar**  
**Cómo reportarlo:** interpretar solo como asociación descriptiva. Usar los términos «individuos por fecha registrada», no «tasa por hora-colector».

## 10. Registros diarios de precipitación

**Pregunta:** ¿qué campos están disponibles en el archivo diario para comparar fechas de muestreo?

In [ ]:
#Importemos la precipitacion diaria entre 2021 y 2023

prep_diaria = pd.read_csv(
    './dhime_diario.csv',
    sep=',',
    encoding='utf-8'
)

prep_diaria.head(5)

**Interpretación / resultados a completar después de ejecutar**  
**Cómo reportarlo:** verificar que no se sumen precipitaciones de estaciones distintas como si fueran una sola estación local.

## 11. Agrupaciones exploratorias de cuatro fechas

**Pregunta:** ¿cómo varían los conteos al reunir cuatro fechas de recolección? Estas agrupaciones NO son campañas comprobadas de cuatro noches.

In [ ]:
# ============================================================
# 17. LÍNEA DE TIEMPO: BATCHES DE 4 JORNADAS DE RECOLECCIÓN
# ============================================================

# ============================================================
# 1. PREPARAR FECHAS
# ============================================================

# Reconstruir la tabla general desde la sección de lluvia/capturas, sin depender de filtros posteriores.
df_mosquitos = df.copy()
col_conteo = 'individualCount'
df_mosquitos[col_conteo] = pd.to_numeric(df_mosquitos[col_conteo], errors='coerce')
df_mosquitos['Fecha_Inicio'] = pd.to_datetime(
    df_mosquitos['eventDate'].astype('string').str.split('/', n=1).str[0],
    errors='coerce', utc=True
)
df_mosquitos = df_mosquitos.dropna(subset=['Fecha_Inicio']).copy()

df_mosquitos['Fecha_Limpia'] = pd.to_datetime(
    df_mosquitos['Fecha_Inicio'],
    errors='coerce',
    utc=True
).dt.tz_convert(None).dt.normalize()

prep_diaria['Fecha_Limpia'] = pd.to_datetime(
    prep_diaria['Fecha'],
    errors='coerce'
).dt.normalize()

prep_diaria['Valor'] = pd.to_numeric(
    prep_diaria['Valor'],
    errors='coerce'
)

# ============================================================
# 2. AGRUPAR MOSQUITOS POR FECHA DE RECOLECCIÓN
# ============================================================

mosquitos_diarios = df_mosquitos.dropna(
    subset=['Fecha_Limpia']
).groupby('Fecha_Limpia').agg(
    Total_Mosquitos=(col_conteo, 'sum')
).reset_index().sort_values('Fecha_Limpia')

# ============================================================
# 3. CREAR BATCHES DE 4 JORNADAS
# ============================================================

mosquitos_diarios['Batch_4J'] = (
    np.arange(len(mosquitos_diarios)) // 4
)

mosquitos_4d = mosquitos_diarios.groupby('Batch_4J').agg(
    Fecha_Inicio=('Fecha_Limpia', 'min'),
    Fecha_Fin=('Fecha_Limpia', 'max'),
    Total_Mosquitos=('Total_Mosquitos', 'sum'),
    Jornadas=('Fecha_Limpia', 'count')
).reset_index()

# ============================================================
# 4. PRECIPITACIÓN SOLO EN LAS FECH4AS DE RECOLECCIÓN
# ============================================================

lluvia_diaria = prep_diaria.dropna(
    subset=['Fecha_Limpia', 'Valor']
).groupby('Fecha_Limpia').agg(
    Precipitacion=('Valor', 'sum')
).reset_index()

# Asociar cada fecha de recolección con su batch
fechas_batch = mosquitos_diarios[
    ['Fecha_Limpia', 'Batch_4J']
].merge(
    lluvia_diaria,
    on='Fecha_Limpia',
    how='left'
)

lluvia_4d = fechas_batch.groupby('Batch_4J').agg(
    Lluvia_Acumulada=('Precipitacion', 'sum'),
    Dias_Con_Registro=('Precipitacion', 'count')
).reset_index()

lluvia_4d.loc[
    lluvia_4d['Dias_Con_Registro'] == 0,
    'Lluvia_Acumulada'
] = np.nan

# ============================================================
# 5. CRUZAR INFORMACIÓN
# ============================================================

df_plot = pd.merge(
    mosquitos_4d,
    lluvia_4d,
    on='Batch_4J',
    how='left'
).sort_values('Fecha_Inicio')

# ============================================================
# 6. GRÁFICA
# ============================================================

fig, ax1 = plt.subplots(figsize=(15, 6))

ax1.plot(
    df_plot['Fecha_Inicio'],
    df_plot['Total_Mosquitos'],
    color='teal',
    marker='o',
    markersize=6,
    linewidth=2,
    label='Mosquitos registrados'
)

ax1.set_xlabel(
    'Fecha de inicio del batch',
    fontsize=12,
    fontweight='bold'
)

ax1.set_ylabel(
    'Mosquitos registrados por 4 jornadas',
    fontsize=12,
    color='teal',
    fontweight='bold'
)

ax1.tick_params(axis='y', labelcolor='teal')

ax1.xaxis.set_major_locator(
    mdates.MonthLocator(interval=2)
)

ax1.xaxis.set_major_formatter(
    mdates.DateFormatter('%b %Y')
)

plt.xticks(rotation=45, ha='right')
ax1.grid(True, linestyle='--', alpha=0.4)

ax2 = ax1.twinx()

ax2.plot(
    df_plot['Fecha_Inicio'],
    df_plot['Lluvia_Acumulada'],
    color='darkorange',
    marker='D',
    markersize=5,
    linewidth=2,
    linestyle='--',
    alpha=0.8,
    label='Precipitación en días de recolección'
)

ax2.set_ylabel(
    'Precipitación acumulada (mm)',
    fontsize=12,
    color='darkorange',
    fontweight='bold'
)

ax2.tick_params(axis='y', labelcolor='darkorange')

# ============================================================
# 7. TÍTULO Y LEYENDAS
# ============================================================

plt.title(
    'Capturas de Mosquitos y Precipitación en Grupos de 4 Jornadas',
    fontsize=15,
    fontweight='bold',
    pad=15
)

handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    handles1 + handles2,
    labels1 + labels2,
    loc='upper left',
    framealpha=0.9
)

plt.tight_layout()
plt.show()

# ============================================================
# 8. RESUMEN
# ============================================================

print("BATCHES TOTALES: ", len(df_plot))


print('\nRESUMEN DE LOS ÚLTIMOS 10 BATCHES:')

print(
    df_plot[
        [
            'Fecha_Inicio',
            'Fecha_Fin',
            'Jornadas',
            'Total_Mosquitos',
            'Lluvia_Acumulada',
            'Dias_Con_Registro'
        ]
    ].tail(10).to_string(index=False)
)

**Interpretación / resultados a completar después de ejecutar**  
**Cómo reportarlo:** excluir agrupaciones incompletas de comparaciones de cuatro fechas y comprobar la separación temporal entre fechas.

## 12. Agrupaciones exploratorias en Cheté

**Pregunta:** ¿qué sucede en la localidad de interés? Comparar con Fermi exige revisar fechas, esfuerzo y definición de conteos.

In [ ]:
# ============================================================
# 17. LÍNEA DE TIEMPO: BATCHES DE 4 JORNADAS - CHETÉ
# ============================================================

# ============================================================
# 0. FILTRAR REGISTROS DE CHETÉ
# ============================================================

df_mosquitos = df_mosquitos[
    df_mosquitos['locality'].astype('string').str.strip().str.casefold() == 'cheté'
].copy()

# ============================================================
# 1. PREPARAR FECHAS
# ============================================================

df_mosquitos['Fecha_Limpia'] = pd.to_datetime(
    df_mosquitos['Fecha_Inicio'],
    errors='coerce',
    utc=True
).dt.tz_convert(None).dt.normalize()

prep_diaria['Fecha_Limpia'] = pd.to_datetime(
    prep_diaria['Fecha'],
    errors='coerce'
).dt.normalize()

prep_diaria['Valor'] = pd.to_numeric(
    prep_diaria['Valor'],
    errors='coerce'
)

# ============================================================
# 2. AGRUPAR MOSQUITOS POR FECHA DE RECOLECCIÓN
# ============================================================

mosquitos_diarios = df_mosquitos.dropna(
    subset=['Fecha_Limpia']
).groupby('Fecha_Limpia').agg(
    Total_Mosquitos=(col_conteo, 'sum')
).reset_index().sort_values('Fecha_Limpia')

# ============================================================
# 3. CREAR BATCHES DE 4 JORNADAS
# ============================================================

mosquitos_diarios['Batch_4J'] = (
    np.arange(len(mosquitos_diarios)) // 4
)

mosquitos_4d = mosquitos_diarios.groupby('Batch_4J').agg(
    Fecha_Inicio=('Fecha_Limpia', 'min'),
    Fecha_Fin=('Fecha_Limpia', 'max'),
    Total_Mosquitos=('Total_Mosquitos', 'sum'),
    Jornadas=('Fecha_Limpia', 'count')
).reset_index()

# ============================================================
# 4. PRECIPITACIÓN SOLO EN LAS FECHAS DE RECOLECCIÓN
# ============================================================

lluvia_diaria = prep_diaria.dropna(
    subset=['Fecha_Limpia', 'Valor']
).groupby('Fecha_Limpia').agg(
    Precipitacion=('Valor', 'sum')
).reset_index()

# Asociar cada fecha de recolección con su batch
fechas_batch = mosquitos_diarios[
    ['Fecha_Limpia', 'Batch_4J']
].merge(
    lluvia_diaria,
    on='Fecha_Limpia',
    how='left'
)

lluvia_4d = fechas_batch.groupby('Batch_4J').agg(
    Lluvia_Acumulada=('Precipitacion', 'sum'),
    Dias_Con_Registro=('Precipitacion', 'count')
).reset_index()

lluvia_4d.loc[
    lluvia_4d['Dias_Con_Registro'] == 0,
    'Lluvia_Acumulada'
] = np.nan

# ============================================================
# 5. CRUZAR INFORMACIÓN
# ============================================================

df_plot = pd.merge(
    mosquitos_4d,
    lluvia_4d,
    on='Batch_4J',
    how='left'
).sort_values('Fecha_Inicio')

# ============================================================
# 6. GRÁFICA
# ============================================================

fig, ax1 = plt.subplots(figsize=(15, 6))

ax1.plot(
    df_plot['Fecha_Inicio'],
    df_plot['Total_Mosquitos'],
    color='teal',
    marker='o',
    markersize=6,
    linewidth=2,
    label='Mosquitos registrados en Cheté'
)

ax1.set_xlabel(
    'Fecha de inicio del batch',
    fontsize=12,
    fontweight='bold'
)

ax1.set_ylabel(
    'Mosquitos registrados por 4 jornadas',
    fontsize=12,
    color='teal',
    fontweight='bold'
)

ax1.tick_params(axis='y', labelcolor='teal')

ax1.xaxis.set_major_locator(
    mdates.MonthLocator(interval=2)
)

ax1.xaxis.set_major_formatter(
    mdates.DateFormatter('%b %Y')
)

plt.xticks(rotation=45, ha='right')
ax1.grid(True, linestyle='--', alpha=0.4)

ax2 = ax1.twinx()

ax2.plot(
    df_plot['Fecha_Inicio'],
    df_plot['Lluvia_Acumulada'],
    color='darkorange',
    marker='D',
    markersize=5,
    linewidth=2,
    linestyle='--',
    alpha=0.8,
    label='Precipitación en días de recolección'
)

ax2.set_ylabel(
    'Precipitación acumulada (mm)',
    fontsize=12,
    color='darkorange',
    fontweight='bold'
)

ax2.tick_params(axis='y', labelcolor='darkorange')

# ============================================================
# 7. TÍTULO Y LEYENDAS
# ============================================================

plt.title(
    'Capturas de Mosquitos en Cheté y Precipitación '
    'en Grupos de 4 Jornadas',
    fontsize=15,
    fontweight='bold',
    pad=15
)

handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    handles1 + handles2,
    labels1 + labels2,
    loc='upper left',
    framealpha=0.9
)

plt.tight_layout()
plt.show()

# ============================================================
# 8. RESUMEN
# ============================================================

print("LOCALIDAD: CHETÉ")
print("REGISTROS:", len(df_mosquitos))
print("JORNADAS DE RECOLECCIÓN:", len(mosquitos_diarios))
print("BATCHES TOTALES:", len(df_plot))

print('\nRESUMEN DE LOS ÚLTIMOS 10 BATCHES:')

print(
    df_plot[
        [
            'Fecha_Inicio',
            'Fecha_Fin',
            'Jornadas',
            'Total_Mosquitos',
            'Lluvia_Acumulada',
            'Dias_Con_Registro'
        ]
    ].tail(10).to_string(index=False)
)

**Interpretación / resultados a completar después de ejecutar**  
**Cómo reportarlo:** comparar tentativamente con Fermi (346), pero solo después de verificar que el esfuerzo y los intervalos de fechas sean comparables.

## 13. Conclusiones y trabajo pendiente

**Conclusiones a completar con las salidas recalculadas:**

1. **Calidad de los registros:** ¿hay suficiente información para trabajar con individuos, fechas y esfuerzo sin asumir valores?
2. **Localidades y especies:** ¿qué diferencias descriptivas aparecen?
3. **Precipitación:** ¿qué patrones se observan y qué explicaciones alternativas existen?
4. **Comparación con Fermi:** ¿qué estimaciones son realmente comparables con 346 *Anopheles* en cuatro noches?

**Pendiente para cumplir íntegramente el reto:** reconstruir campañas reales si existen identificadores suficientes, reservar campañas de evaluación final, estimar incertidumbre (p. ej., bootstrap agrupado), entrenar un modelo sencillo y reportar su MAE frente al promedio histórico y Fermi revisada. No se han realizado estas etapas en este notebook.

**Advertencia:** fechas de inicio de intervalos, número de ocurrencias, conteos de individuos y campañas son unidades distintas; no deben intercambiarse. Si `individualCount` contiene valores faltantes, reportar cuántos son antes de calcular sumas y decidir justificadamente su tratamiento.